# 최종 제출 파일 재현 노트

`sub_v44_anti_v41layout035_a120.csv`를 다시 만드는 검증용 노트북입니다.

- 최종 파일: `sub_v44_anti_v41layout035_a120.csv`
- Public Score: `9.6773143392`
- 평가 지표: MAE
- 빠른 재현: `artifacts/`에 포함된 CSV로 최종 산식 재계산
- 전체 재현: 공식 데이터를 `data/`에 둔 뒤 마지막 pipeline 셀 실행


## 규칙 관련 정리

- 평가 데이터의 target은 어떤 단계에서도 사용하지 않았습니다.
- test prediction을 label로 사용하는 pseudo labeling은 수행하지 않았습니다.
- 모델 학습 및 잔차 보정은 train target과 train OOF prediction을 기준으로 수행했습니다.
- test 데이터는 동일한 feature engineering pipeline을 통한 inference matrix 생성 및 최종 예측 산출에만 사용했습니다.
- A120은 저장된 prediction CSV를 산식대로 조합한 후처리 결과입니다.


## 사용한 파일 흐름

| 단계 | 파일 | 산출물 / 역할 |
|---|---|---|
| 피처 생성 | `01_features.py` | `features.npz` 생성 |
| 기본 앙상블 | `07_v12_full.py` | `v12_full_oof_bundle.npz`, v12 prediction |
| Future proxy 1 | `21_future_proxy_model.py` | `future_proxy_oof.npz` |
| Future proxy 2 | `23_xgb_future_proxy.py` | `xgb_future_proxy_oof.npz` |
| Future proxy 3 | `24_future_proxy_allcols.py` | `future_proxy_allcols_oof.npz` |
| Future stack | `final_pipeline/compact_final_stack.py` | `final_future_stack_oof.npz`, `sub_v28_future_stack_scaled.csv` |
| Blend | `final_pipeline/portfolio_blends.py` | `sub_port_v28_w70.csv` |
| OOF 잔차 보정 | `final_pipeline/w70_residual_calibration.py` | v40 보정 후보 |
| 조합 후보 생성 | `final_pipeline/w70_combo_candidates.py` | `sub_v41_rank_layout_a035_add_w65.csv` |
| 최종 후처리 | `final_pipeline/late_probe_postprocess.py` | `sub_v44_anti_v41layout035_a120.csv` |


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
ARTIFACT_DIR = ROOT / "artifacts"
TARGET = "avg_delay_minutes_next_30m"

print("Python:", sys.version.split()[0])
print("Working directory:", ROOT)


## 1. 필요한 artifact 확인

아래 3개 CSV가 있으면 최종 제출 파일을 빠르게 다시 만들 수 있습니다.

- `artifacts/sub_port_v28_w70.csv`
- `artifacts/sub_v41_rank_layout_a035_add_w65.csv`
- `artifacts/sub_v44_anti_v41layout035_a120.csv`

In [ ]:
required_artifacts = {
    "base_w70": ARTIFACT_DIR / "sub_port_v28_w70.csv",
    "failed_v41": ARTIFACT_DIR / "sub_v41_rank_layout_a035_add_w65.csv",
    "final_a120_reference": ARTIFACT_DIR / "sub_v44_anti_v41layout035_a120.csv",
}

missing = [str(path) for path in required_artifacts.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required artifact files:\n" + "\n".join(missing))

for name, path in required_artifacts.items():
    print(f"{name:22s} {path}  ({path.stat().st_size:,} bytes)")


## 2. artifact로 최종 제출 파일 재생성

최종 산식은 다음과 같습니다.

```text
A120 = sub_port_v28_w70
       - 1.20 * (sub_v41_rank_layout_a035_add_w65 - sub_port_v28_w70)
```

`sub_v41_rank_layout_a035_add_w65.csv`가 Public에서 악화되어, 같은 방향을 반대로 1.20배 적용했습니다.

In [ ]:
base = pd.read_csv(required_artifacts["base_w70"])
failed = pd.read_csv(required_artifacts["failed_v41"])
reference = pd.read_csv(required_artifacts["final_a120_reference"])

base_pred = base[TARGET].to_numpy(dtype=np.float64)
failed_pred = failed[TARGET].to_numpy(dtype=np.float64)

a120_pred = np.clip(base_pred - 1.20 * (failed_pred - base_pred), 0, None)
a120_pred += base_pred.mean() - a120_pred.mean()
a120_pred = np.clip(a120_pred, 0, None).astype(np.float32)

out = base.copy()
out[TARGET] = a120_pred
output_path = ROOT / "sub_v44_anti_v41layout035_a120_reproduced_from_notebook.csv"
out.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(out))
print("Mean:", float(out[TARGET].mean()))
print("Std:", float(out[TARGET].std()))


## 3. 저장된 최종 파일과 비교

CSV 저장 과정의 반올림 때문에 아주 작은 차이는 날 수 있습니다. 아래 차이는 점수에 영향이 없는 수준입니다.

In [ ]:
ref_pred = reference[TARGET].to_numpy(dtype=np.float64)
diff = ref_pred - a120_pred.astype(np.float64)

summary = pd.DataFrame(
    {
        "metric": ["max_abs_diff", "mean_abs_diff", "reference_mean", "reproduced_mean"],
        "value": [
            float(np.max(np.abs(diff))),
            float(np.mean(np.abs(diff))),
            float(ref_pred.mean()),
            float(a120_pred.mean()),
        ],
    }
)
summary


## 4. 최종 선택에 참고한 Public 제출 이력

아래 점수들은 Public leaderboard에서 확인한 제출 이력입니다.

In [ ]:
history = pd.DataFrame(
    [
        ["sub_v28_v1265_fp35.csv", 9.6855430648, "v12/future blend"],
        ["sub_v28_v1260_fp40.csv", 9.6837787747, "v12/future blend"],
        ["sub_port_v28_w70.csv", 9.6799310594, "portfolio base"],
        ["sub_v41_rank_layout_a035_add_w65.csv", 9.6860945236, "Public에서 악화된 보정 방향"],
        ["sub_v42_anti_v41layout035_a085.csv", 9.6774917384, "반대 방향 0.85배"],
        ["sub_v42_anti_v41layout035_a100.csv", 9.6773562672, "반대 방향 1.00배"],
        ["sub_v44_anti_v41layout035_a120.csv", 9.6773143392, "final A120"],
    ],
    columns=["file", "public_mae", "note"],
)
history


## 5. 전체 파이프라인 실행용 셀

아래 셀은 공식 데이터를 `data/` 폴더에 넣은 뒤 전체 pipeline을 실행할 때 사용합니다.

주의: 전체 재학습은 GPU와 라이브러리 버전에 따라 미세한 수치 차이가 날 수 있습니다. 최종 파일 확인은 위 artifact 기반 재현을 사용했습니다.

In [ ]:
RUN_FULL_PIPELINE = False

pipeline_commands = [
    [sys.executable, "01_features.py"],
    [sys.executable, "07_v12_full.py"],
    [sys.executable, "21_future_proxy_model.py"],
    [sys.executable, "23_xgb_future_proxy.py"],
    [sys.executable, "24_future_proxy_allcols.py"],
    [sys.executable, "final_pipeline/compact_final_stack.py"],
    [sys.executable, "final_pipeline/portfolio_blends.py"],
    [sys.executable, "final_pipeline/w70_residual_calibration.py"],
    [sys.executable, "final_pipeline/w70_combo_candidates.py"],
    [sys.executable, "final_pipeline/late_probe_postprocess.py"],
]

if RUN_FULL_PIPELINE:
    import subprocess
    for cmd in pipeline_commands:
        print("Running:", " ".join(cmd))
        subprocess.run(cmd, check=True)
else:
    print("RUN_FULL_PIPELINE=False 입니다. 전체 학습을 실행하려면 True로 바꾸세요.")
    for cmd in pipeline_commands:
        print(" ".join(cmd))


## 6. Environment

검증 당시 사용한 주요 라이브러리 버전입니다.

In [ ]:
packages = ["numpy", "pandas", "sklearn", "scipy", "lightgbm", "xgboost", "catboost"]
for pkg in packages:
    try:
        module = __import__(pkg)
        print(f"{pkg:12s} {getattr(module, '__version__', 'unknown')}")
    except Exception as exc:
        print(f"{pkg:12s} NOT INSTALLED ({exc})")
